# AI-RAG · Week 11 — Notebook 1
## The Four Metrics & RAGAS

*The measure-it week, made runnable.* This notebook walks the four RAG-specific metrics — **faithfulness, answer relevance, context precision, context recall** — the golden set they score against, the row-builder that feeds them, and the task-success judge. Every cell runs **offline** (deterministic token-overlap *proxies*, no API key, no cost) so you can run it top-to-bottom while you explain it. Set an `OPENAI_API_KEY` and remove the `EVAL_OFFLINE` line in setup to switch to **live RAGAS**.

> The code below is lifted **verbatim** from the package modules (`src/eval/*.py`). The only changes are notebook glue (paths, and dropping the relative-import lines because the code is defined inline here).

### Notebook Setup – Dependencies, Golden Set and Evaluation Mode

This cell prepares the notebook environment before starting the RAG evaluation.

- Checks whether Pydantic is installed and installs it if necessary.
- Uses the existing CBSE Class 9 golden-set file at `golden_set/capstone_v1.jsonl`.
- Checks that the golden-set file exists and counts its active records.
- Sets `EVAL_OFFLINE = 1` to enable offline evaluation without requiring an API key.

**Important:** The existing golden-set file is used directly. This cell must not overwrite or recreate it.

In [1]:

# --- notebook setup ---------------------------------------------------------

# 1) Ensure pydantic is available
# 2) Use the existing CBSE golden set
# 3) Force the OFFLINE evaluation path

try:
    import pydantic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pydantic"]
    )

import os
from pathlib import Path

# Force offline evaluation
os.environ["EVAL_OFFLINE"] = "1"

# Locate the existing CBSE golden set
golden_path = Path("golden_set/capstone_v1.jsonl")

if not golden_path.exists():
    raise FileNotFoundError(
        f"CBSE golden set not found at: {golden_path}"
    )

# Count active records (ignore blank lines and // comments)
entry_count = sum(
    1 for line in golden_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.strip().startswith("//")
)

print(
    "Using existing golden set:",
    entry_count,
    "entries · EVAL_OFFLINE =",
    os.environ["EVAL_OFFLINE"]
)


Using existing golden set: 20 entries · EVAL_OFFLINE = 1


## 1 · The Golden Set – Loader and Validator

This cell defines the functions and data model used to load and validate the golden-set JSONL file.

It contains three main components:

1. **GoldenEntry:** A Pydantic model that defines the expected structure of each golden-set record. It checks the ID format, question, ideal answer and notes.
2. **load_golden_set():** Reads the JSONL file one line at a time, converts each valid JSON record into a GoldenEntry object and skips blank lines or comment lines.
3. **validate_golden_set():** Checks for duplicate IDs and classifies questions into happy, harder and edge categories using the notes field.

The `ground_truth` property makes the ideal answer available in the string format expected by RAGAS.

The loader reads `golden_set/capstone_v1.jsonl`, which currently contains our 20 CBSE Class 9 questions.

In [2]:
"""src/eval/golden.py — loader and validator for the capstone golden set.

Lineage: carried forward unchanged from W5. The golden set is the ground truth
that every later week's KPI is computed against — treat it as infrastructure,
not a one-off fixture. As of W10 it lives at `golden_set/capstone_v1.jsonl`
(the W5 set plus W7/W9 additions), 20 entries, ids g001..g020.

A golden entry has four fields:
  • id            — stable identifier (g001..g020). Referenced in eval runs
                    and across docs.
  • question      — exactly as a real user would phrase it.
  • ideal_answer  — full reference text. RAGAS uses this as `ground_truth`;
                    the judge uses it as the ideal answer. For open-ended
                    questions this may be a properties dict.
  • notes         — category (happy/harder/edge), source doc, why it matters.
"""
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from pydantic import BaseModel, Field, field_validator

_ID_PATTERN = re.compile(r"^g\d{3}$")


class GoldenEntry(BaseModel):
    """One row of the golden set."""

    id: str = Field(..., description="Stable identifier g001..g999")
    question: str = Field(..., min_length=1)
    ideal_answer: str | dict[str, Any] = Field(...)
    notes: str = Field(default="", description="Category, source, why this matters")

    @field_validator("id")
    @classmethod
    def _id_shape(cls, v: str) -> str:
        if not _ID_PATTERN.match(v):
            raise ValueError(f"id must match g001..g999; got {v!r}")
        return v

    @property
    def ground_truth(self) -> str:
        """RAGAS expects a string ground_truth. Flatten dict ideals to text."""
        if isinstance(self.ideal_answer, dict):
            return json.dumps(self.ideal_answer)
        return self.ideal_answer


def load_golden_set(path: str | Path) -> list[GoldenEntry]:
    """Read a .jsonl file. Each non-blank, non-comment line is one entry."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Golden set not found at {p}")
    entries: list[GoldenEntry] = []
    with p.open() as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line or line.startswith("//"):
                continue
            try:
                data = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"{p}:{lineno} — malformed JSON: {e.msg}") from e
            entries.append(GoldenEntry.model_validate(data))
    return entries


def validate_golden_set(path: str | Path) -> dict[str, Any]:
    """Validate the file. Returns a summary dict; raises on errors."""
    entries = load_golden_set(path)

    ids = [e.id for e in entries]
    if len(ids) != len(set(ids)):
        seen: set[str] = set()
        dupes = [i for i in ids if i in seen or seen.add(i)]  # noqa: B015
        raise ValueError(f"Duplicate ids in {path}: {sorted(set(dupes))}")

    def _bucket(note: str) -> str:
        low = note.lower()
        if low.startswith("happy"):
            return "happy"
        if low.startswith("harder") or low.startswith("multi"):
            return "harder"
        if low.startswith("edge"):
            return "edge"
        return "uncategorised"

    counts: dict[str, int] = {}
    for e in entries:
        counts[_bucket(e.notes)] = counts.get(_bucket(e.notes), 0) + 1

    return {"path": str(path), "n": len(entries), "ids": ids, "coverage": counts}


### Load and Validate the CBSE Golden Set

This cell uses the loader and validator functions defined above to read and inspect our CBSE Class 9 golden set.

- Loads all records from `golden_set/capstone_v1.jsonl`.
- Checks the number of records and verifies that there are no duplicate IDs.
- Calculates question coverage across the happy, harder and edge categories using the `notes` field.
- Displays the first question to confirm that the CBSE dataset has been loaded.

**Expected result:** 20 golden-set entries with IDs `g001` to `g020`.

In [3]:
golden = load_golden_set('golden_set/capstone_v1.jsonl')
summary = validate_golden_set('golden_set/capstone_v1.jsonl')
print('entries:', summary['n'])
print('coverage:', summary['coverage'])
print('first:', golden[0].id, '—', golden[0].question)

entries: 20
coverage: {'happy': 14, 'harder': 4, 'edge': 2}
first: g001 — What do R1, R2, and R3 mean in the revised language scheme?


### Golden Set Validation – Output

The CBSE Class 9 golden set was loaded successfully with 20 entries.

The validation summary shows:
- 14 happy-path questions
- 4 harder questions
- 2 edge questions

All 20 records were loaded, and the first record displayed is the CBSE language-scheme question (`g001`).

This confirms that the notebook is using the intended CBSE golden set with the expected record structure and category distribution.

## 2.  The Four RAGAS Metrics – Two Evaluation Modes

This cell defines four RAG-specific evaluation metrics used to measure the quality of a RAG system.

**The four metrics:**

1. **Faithfulness:** Checks whether the generated answer is supported by the retrieved context.
2. **Answer Relevance:** Checks whether the generated answer addresses the user's question.
3. **Context Precision:** Measures how relevant the retrieved chunks are to the expected answer.
4. **Context Recall:** Measures how much of the expected answer's information is present in the retrieved context.

**Two evaluation modes:**

- **Offline mode:** Uses deterministic token-overlap calculations as proxy metrics. It does not require an API key or incur API costs. These scores are not actual RAGAS scores.
- **Live mode:** Uses the RAGAS framework to calculate the metrics with an LLM judge.

The cell defines a `MetricResult` data class to store the metric name, evaluation mode, aggregate score and individual question scores.

It also defines helper functions for token processing, offline proxy calculations, live RAGAS evaluation and scoring all four metrics.

The actual evaluation will happen later, after the evaluation rows have been constructed.

In [4]:
"""src/eval/ragas_metrics.py — the four RAG-specific metrics, two ways to run.

W11 standardises on RAGAS for the four RAG metrics:
  • faithfulness        — is the answer supported by the retrieved context?
  • answer_relevance    — does the answer address the question?
  • context_precision   — are the top-K retrieved chunks the right ones?
  • context_recall      — did we retrieve everything we needed?

Each public function takes a list of `rows` and returns a `MetricResult`
(per-row scores + an aggregate). A row is a dict with:
    {"id", "question", "answer", "contexts": [str, ...], "ground_truth"}

TWO MODES
---------
LIVE  (default when an API key is set and `ragas` is importable):
      real RAGAS metrics, each backed by an LLM judge call per row. This is
      what produces the numbers you put in the snapshot and defend at M2.

OFFLINE (set EVAL_OFFLINE=1, or when `ragas` isn't installed):
      deterministic token-overlap *proxies*. These are NOT RAGAS numbers —
      they exist so the eval suite runs in CI / on a laptop with no API key,
      proving the wiring is sound. Every offline MetricResult is labelled
      `mode="offline-proxy"` so a proxy number is never mistaken for a real
      RAGAS score. The smoke test runs this path.

This split is the whole point of "eval as code": the suite is runnable and
assertable on every commit (offline), and produces real numbers on demand
(live). See docs/eval/README.md.
"""
from __future__ import annotations

import os
import re
from dataclasses import dataclass, field


# ─── Result container ────────────────────────────────────────────────────────
@dataclass
class MetricResult:
    """One metric's outcome across all rows."""

    name: str
    mode: str                       # "live" or "offline-proxy"
    aggregate: float                # mean across rows, 0..1
    per_row: dict[str, float] = field(default_factory=dict)  # id -> score

    def __str__(self) -> str:
        return f"{self.name}={self.aggregate:.3f} ({self.mode}, n={len(self.per_row)})"


# ─── Mode detection ──────────────────────────────────────────────────────────
def offline_mode() -> bool:
    """Offline if explicitly forced, or if ragas / an API key is unavailable."""
    if os.getenv("EVAL_OFFLINE", "").strip() in {"1", "true", "TRUE", "yes"}:
        return True
    if not os.getenv("OPENAI_API_KEY"):
        return True
    try:
        import ragas  # noqa: F401
    except Exception:
        return True
    return False


# ─── Token helpers (offline proxy) ───────────────────────────────────────────
_STOP = {
    "the", "a", "an", "and", "or", "but", "if", "of", "to", "in", "on", "for",
    "with", "is", "are", "was", "were", "be", "been", "as", "at", "by", "it",
    "this", "that", "these", "those", "from", "your", "you", "may", "must",
    "can", "do", "does", "how", "what", "who", "i", "my", "me", "we", "our",
    "per", "up", "any", "all", "via", "not", "no", "they", "their",
}


def _tokens(text: str) -> set[str]:
    if not text:
        return set()
    raw = re.findall(r"[a-z0-9]+", text.lower())
    return {t for t in raw if len(t) > 2 and t not in _STOP}


def _overlap_fraction(a: set[str], b: set[str]) -> float:
    """Fraction of `a`'s tokens that appear in `b`. 0..1. Empty `a` -> 0.0."""
    if not a:
        return 0.0
    return len(a & b) / len(a)


# ─── Offline proxies (deterministic, dependency-free) ────────────────────────
def _proxy(name: str, rows: list[dict], fn) -> MetricResult:
    per_row = {r["id"]: round(fn(r), 4) for r in rows}
    agg = round(sum(per_row.values()) / len(per_row), 4) if per_row else 0.0
    return MetricResult(name=name, mode="offline-proxy", aggregate=agg, per_row=per_row)


def _faithfulness_proxy(r: dict) -> float:
    # How much of the answer is covered by the retrieved context?
    ans = _tokens(r.get("answer", ""))
    ctx = _tokens(" ".join(r.get("contexts", [])))
    return _overlap_fraction(ans, ctx)


def _answer_relevance_proxy(r: dict) -> float:
    # Does the answer address the question? Blend (a) how many of the question's
    # key terms the answer covers with (b) whether the answer engages the topic
    # at all. Short questions make pure token-overlap jumpy, so the topical-touch
    # term stabilises a correct answer near ~0.7 while a truly off-topic answer
    # (no shared terms) still collapses to 0.0.
    q = _tokens(r.get("question", ""))
    ans = _tokens(r.get("answer", ""))
    if not q:
        return 0.0
    coverage = _overlap_fraction(q, ans)            # 0..1
    touches = 1.0 if (q & ans) else 0.0             # any shared key term?
    return round(0.5 * coverage + 0.5 * touches, 4)


def _context_precision_proxy(r: dict) -> float:
    # Of the retrieved chunks, what fraction are relevant to the ground truth?
    gt = _tokens(r.get("ground_truth", ""))
    contexts = r.get("contexts", []) or []
    if not contexts:
        return 0.0
    relevant = sum(1 for c in contexts if _tokens(c) & gt)
    return relevant / len(contexts)


def _context_recall_proxy(r: dict) -> float:
    # How much of the ground truth is present across the retrieved chunks?
    gt = _tokens(r.get("ground_truth", ""))
    ctx = _tokens(" ".join(r.get("contexts", [])))
    return _overlap_fraction(gt, ctx)


# ─── Live RAGAS ──────────────────────────────────────────────────────────────
def _ragas_score(name: str, rows: list[dict], metric, column: str) -> MetricResult:
    """Run one RAGAS metric over the rows. Imported lazily (heavy deps).

    Targets ragas ~0.1.x (see requirements.txt for the pin). RAGAS has changed
    column names and metric imports across versions — if this raises, check the
    pinned version and the Lab Guide's 'RAGAS version pinning' note.
    """
    from datasets import Dataset
    from ragas import evaluate

    ds = Dataset.from_dict({
        "question":     [r["question"] for r in rows],
        "answer":       [r["answer"] for r in rows],
        "contexts":     [list(r.get("contexts", [])) for r in rows],
        "ground_truth": [r.get("ground_truth", "") for r in rows],
    })
    result = evaluate(ds, metrics=[metric])
    df = result.to_pandas()

    per_row = {
        rows[i]["id"]: round(float(df[column].iloc[i]), 4)
        for i in range(len(rows))
    }
    agg = round(float(df[column].mean()), 4)
    return MetricResult(name=name, mode="live", aggregate=agg, per_row=per_row)


# ─── Public API — one function per metric ────────────────────────────────────
def score_faithfulness(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("faithfulness", rows, _faithfulness_proxy)
    from ragas.metrics import faithfulness
    return _ragas_score("faithfulness", rows, faithfulness, "faithfulness")


def score_answer_relevance(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("answer_relevance", rows, _answer_relevance_proxy)
    from ragas.metrics import answer_relevancy
    return _ragas_score("answer_relevance", rows, answer_relevancy, "answer_relevancy")


def score_context_precision(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("context_precision", rows, _context_precision_proxy)
    from ragas.metrics import context_precision
    return _ragas_score("context_precision", rows, context_precision, "context_precision")


def score_context_recall(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("context_recall", rows, _context_recall_proxy)
    from ragas.metrics import context_recall
    return _ragas_score("context_recall", rows, context_recall, "context_recall")


# Convenience: all four at once (used by run_eval.py and the baseline doc).
ALL_METRICS = (
    score_faithfulness,
    score_answer_relevance,
    score_context_precision,
    score_context_recall,
)


def score_all(rows: list[dict]) -> dict[str, MetricResult]:
    results = [fn(rows) for fn in ALL_METRICS]   # one call each (matters live!)
    return {r.name: r for r in results}


## 3 · Build the rows — `src/eval/harness.py`

The harness builds the rows one way offline (a deterministic synthetic pipeline) and another way live (your real capstone). A few rows are **deliberately degraded** so the report has real signal: `g019` hallucinates, `g015`/`g005` suffer retrieval misses.

In [5]:
"""src/eval/harness.py — build the eval rows once, two ways.

Shared by eval/conftest.py (the pytest suite) and scripts/run_eval.py (the CLI),
so the offline synthetic pipeline and the live capstone pipeline are defined in
exactly one place.

Row shape (identical offline and live):
    {"id", "question", "answer", "contexts": [str,...], "ground_truth",
     "cache_hit", "latency_ms", "cost_usd", "notes"}
"""
from __future__ import annotations

from pathlib import Path

# GoldenEntry, load_golden_set, offline_mode are defined in the cells above

REPO_ROOT = Path.cwd()   # notebook: resolve paths from the working directory
GOLDEN_PATH = REPO_ROOT / "golden_set" / "capstone_v1.jsonl"

# KB version active at eval time. Live mode reads it from the pipeline; offline
# pins the W10 value so every eval run and snapshot is tagged by KB state (D4).
OFFLINE_KB_VERSION = "v2.w10"

# A clean distractor sharing almost no content tokens with policy answers, so
# context precision reflects relevant-vs-retrieved honestly.

_DISTRACTOR = (
    "The weather forecast predicts moderate rainfall and cloudy skies "
    "during the coming week."
)

def _split_two(text: str) -> tuple[str, str]:
    words = text.split()
    mid = max(1, len(words) // 2)
    return " ".join(words[:mid]), " ".join(words[mid:])




def _synth_row(entry: GoldenEntry, index: int) -> dict:
    """Build one deterministic synthetic evaluation row.

    Most questions receive an ideal answer supported by two relevant
    context chunks and one unrelated distractor.

    Three CBSE-specific failures are deliberately injected:
    - g005: partial retrieval
    - g015: partial retrieval
    - g019: unsupported school-specific examination dates

    These are synthetic test cases, not actual CBSE RAG results.
    """
    gt = entry.ground_truth

    # Divide the ideal answer into two context chunks
    c1, c2 = _split_two(gt)

    # Default: two relevant chunks and one unrelated distractor
    contexts = [c1, c2, _DISTRACTOR]
    answer = gt

    # CBSE failure 1: g005 — partial retrieval
    if entry.id == "g005":
        contexts = [c1, _DISTRACTOR, _DISTRACTOR]

    # CBSE failure 2: g015 — partial retrieval
    elif entry.id == "g015":
        contexts = [c1, _DISTRACTOR, _DISTRACTOR]

    # CBSE failure 3: g019 — deliberately unsupported answer
    elif entry.id == "g019":
        answer = (
            "The 2026–27 Class IX final examinations at your school "
            "are scheduled from 10 March to 20 March 2027."
        )
        contexts = [_DISTRACTOR, _DISTRACTOR, _DISTRACTOR]

    # Retain the original cache and performance simulation
    cache_hit = index < 13

    return {
        "id":           entry.id,
        "question":     entry.question,
        "answer":       answer,
        "contexts":     contexts,
        "ground_truth": gt,
        "cache_hit":    cache_hit,
        "latency_ms":   184 if cache_hit else 785,
        "cost_usd":     0.000041 if cache_hit else 0.000412,
        "notes":        entry.notes,
    }


def _live_rows(golden: list[GoldenEntry]) -> list[dict]:  # pragma: no cover - live only
    """Run the real capstone over the golden set. Needs the learner's src/rag."""
    try:
        from src.rag.qdrant_rag import ask_rag
        from src.rag.qdrant_store import QdrantStore
        from src.rag.retrieval import (
            build_bm25_index,
            hybrid_retrieve,
            load_chunks_from_qdrant,
        )
    except Exception as exc:
        raise RuntimeError(
            "Live eval needs your capstone code (src/rag/*) on the path and "
            "OPENAI_API_KEY + QDRANT_URL set. Run offline with EVAL_OFFLINE=1. "
            f"Import error: {exc}"
        ) from exc

    store = QdrantStore()
    chunks = load_chunks_from_qdrant(store)          # tombstone-filtered (W10)
    bm25 = build_bm25_index(chunks)
    rows: list[dict] = []
    for e in golden:
        res = ask_rag(e.question, store, k=3)
        # Full chunk text for RAGAS contexts comes from retrieval, not from
        # ask_rag's stripped citations (D3).
        retrieved = hybrid_retrieve(store, bm25, chunks, e.question, top_k=3)
        rows.append({
            "id":           e.id,
            "question":     e.question,
            "answer":       res["answer"],
            "contexts":     [c.get("text", "") for c in retrieved],
            "ground_truth": e.ground_truth,
            "cache_hit":    res.get("cache_hit", False),
            "latency_ms":   res.get("latency_ms", 0),
            "cost_usd":     res.get("cost_usd", 0.0),
            "notes":        e.notes,
        })
    return rows


def active_kb_version() -> str:
    if offline_mode():
        return OFFLINE_KB_VERSION
    try:  # pragma: no cover - live only
        from src.rag.pipeline import KB_VERSION
        return KB_VERSION
    except Exception:  # pragma: no cover
        return "unknown"


def load_golden() -> list[GoldenEntry]:
    return load_golden_set(GOLDEN_PATH)


def build_rows(golden: list[GoldenEntry] | None = None) -> list[dict]:
    """The (question, answer, contexts, ground_truth, ...) rows under test."""
    golden = golden or load_golden()
    if offline_mode():
        return [_synth_row(e, i) for i, e in enumerate(golden)]
    return _live_rows(golden)


### Evaluation Harness – CBSE Adaptation

The evaluation harness has been adapted to use the CBSE Class IX golden set.

For offline evaluation:

- Most questions use the ideal answer as the synthetic answer.
- The ideal answer is divided into two context chunks.
- One unrelated distractor chunk is added.
- Three CBSE-specific failure examples are deliberately injected:
  - **g005:** Partial retrieval of the expected information.
  - **g015:** Partial retrieval of the language-scheme requirements.
  - **g019:** An unsupported answer invents school-specific examination dates.
- The original cache-hit, latency and cost simulation is retained.

These are controlled synthetic examples used to test the evaluation workflow. They are not actual responses or retrieval results from the CBSE RAG application.

The live evaluation path remains separate and uses the actual capstone application.

### Build the CBSE Evaluation Rows

This cell calls `build_rows()` to create one evaluation row for each of the 20 CBSE golden-set questions.

It then displays:
- Total number of generated rows
- Active knowledge-base version
- One sample evaluation row
- A few selected CBSE records for inspection

The rows are synthetic offline evaluation data. They are not actual responses or retrieval results from the CBSE RAG application.

In [6]:

rows = build_rows()

print("Total rows:", len(rows))
print("KB version:", active_kb_version())

print("\n--- FAILURE CASE INSPECTION ---")

for r in rows:
    if r["id"] in ("g005", "g015", "g019"):
        print("\nID:", r["id"])
        print("Question:", r["question"])
        print("Answer:", r["answer"])
        print("Ground truth:", r["ground_truth"])
        print("Retrieved contexts:")
        for i, context in enumerate(r["contexts"], 1):
            print(f"  Chunk {i}: {context}")


Total rows: 20
KB version: v2.w10

--- FAILURE CASE INSPECTION ---

ID: g005
Question: Which portal or system is mentioned for updating R3 language information?
Answer: The curriculum refers to the OASIS portal for updating R3 language information.
Ground truth: The curriculum refers to the OASIS portal for updating R3 language information.
Retrieved contexts:
  Chunk 1: The curriculum refers to the OASIS
  Chunk 2: The weather forecast predicts moderate rainfall and cloudy skies during the coming week.
  Chunk 3: The weather forecast predicts moderate rainfall and cloudy skies during the coming week.

ID: g015
Question: A teacher wants to plan Class IX language study while complying with the revised scheme. What two requirements should the teacher check in the curriculum documents before advising students?
Answer: The teacher should check the definitions and requirements for R1, R2, and R3, including the requirement concerning native Indian languages, and should apply the specific con

### Evaluation Rows – Output

The evaluation harness successfully generated 20 rows from the CBSE Class 9 golden set.

Each row contains the question, synthetic answer, three context chunks, ground-truth answer and evaluation metadata.

The active knowledge-base version is `v2.w10`.

The selected records confirm that the generated rows use CBSE questions, including the out-of-corpus school-specific examination-date question (`g019`).

These are offline synthetic evaluation rows, not actual RAG application results.

## 4 · Score the golden set

Now that we have both the metric functions and the rows, score them. **Faithfulness across the 20 questions** — then read the lowest rows (the seeded dips should surface):

In [7]:
faith = score_faithfulness(rows)
print(faith)   # name=..., mode=offline-proxy, aggregate, n

print('\nlowest-scoring rows:')
for rid, sc in sorted(faith.per_row.items(), key=lambda kv: kv[1])[:5]:
    print(f'  {rid}: {sc:.3f}')

faithfulness=0.898 (offline-proxy, n=20)

lowest-scoring rows:
  g019: 0.000
  g005: 0.429
  g015: 0.538
  g001: 1.000
  g002: 1.000


### Faithfulness Evaluation Results

The faithfulness metric measures how much of an answer is supported by the retrieved context.

The offline evaluation produced an aggregate faithfulness score of **0.898 across 20 questions**.

The five lowest-scoring rows were:

| Question ID | Faithfulness | Observation |
|---|---:|---|
| g019 | 0.000 | Unsupported answer containing invented examination dates |
| g005 | 0.429 | Partial retrieval of the OASIS portal information |
| g015 | 0.538 | Partial retrieval of the language-scheme requirements |
| g001 | 1.000 | Full token support |
| g002 | 1.000 | Full token support |

The three deliberately injected failure cases appear at the bottom of the faithfulness results.

This confirms that the evaluation workflow can identify unsupported answers and incomplete retrieved context.

**Note:** These are deterministic offline token-overlap proxy scores, not live RAGAS scores or measurements of the actual CBSE RAG application's performance.

### Calculate All Four Evaluation Metrics

This cell calls `score_all()` to calculate all four RAG evaluation metrics for the 20 golden-set questions.

The four metrics are:

1. **Faithfulness:** Is the answer supported by the supplied context?
2. **Answer relevance:** Does the answer address the question?
3. **Context precision:** Are the retrieved context chunks relevant?
4. **Context recall:** Do the retrieved chunks contain the information needed to answer the question?

The function returns one result for each metric, including its aggregate score and evaluation mode.

In this notebook, the metrics use the offline-proxy method. These scores demonstrate the evaluation workflow and are not actual live RAGAS results.

In [8]:
results = score_all(rows)
for name, res in results.items():
    print(f'{name:20s} {res.aggregate:.3f}   ({res.mode})')

faithfulness         0.898   (offline-proxy)
answer_relevance     0.795   (offline-proxy)
context_precision    0.600   (offline-proxy)
context_recall       0.898   (offline-proxy)


### Four-Metric Evaluation Results

The four offline evaluation metrics were calculated across all 20 CBSE golden-set questions.

| Metric | Aggregate score | What it measures |
|---|---:|---|
| Faithfulness | 0.898 | Whether the answer is supported by retrieved context |
| Answer relevance | 0.795 | Whether the answer addresses the question |
| Context precision | 0.600 | How many retrieved chunks are relevant |
| Context recall | 0.898 | How much of the expected answer is covered by retrieved context |

The results demonstrate two evaluation perspectives:

- **Answer evaluation:** Faithfulness and answer relevance inspect the generated answer.
- **Retrieval evaluation:** Context precision and context recall inspect the retrieved context.

The lower context precision reflects the unrelated distractor chunks and the deliberately injected retrieval failures.

The three seeded failure cases, g005, g015 and g019, demonstrate how incomplete retrieval and unsupported answers affect evaluation scores.

**Note:** These are deterministic offline token-overlap proxy results, not live RAGAS measurements or actual application performance results.

## 5 · The task-success judge — `src/eval/judge.py`

From W5, carried forward. RAGAS now owns the four RAG metrics; this LLM-as-judge is repurposed as the **task-success grader for KPI #1** — *did the answer meet the user's goal end-to-end?* — on a 3-dimension rubric (accuracy / groundedness / format, 1–4 each). It's **live-only** (needs an API key), so below we show the code and exercise its scoring logic locally **without** an API call.

In [9]:
"""src/eval/judge.py — LLM-as-judge (task-success grader).

Lineage: this is the W5 judge, carried forward into the capstone and lightly
hardened for W11. In W5 it was the headline eval tool; from W11 onwards RAGAS
owns the four RAG-specific metrics (faithfulness, answer relevance, context
precision, context recall) and this judge is repurposed as the *task-success*
grader for KPI #1 — "did the answer actually meet the user's goal end-to-end".

Two changes vs the W5 reference:
  1. Lazy client — we never instantiate AsyncOpenAI at import time, so the
     module imports cleanly in pytest with no API key set (W7/W8 footgun).
  2. A thin sync wrapper `judge_one_sync` so the eval suite and KPI step can
     call the judge without managing an event loop.

The 3-dimension rubric (accuracy / groundedness / format, 1-4 each) is the
W5 rubric, unchanged. Adapt the dimension definitions if your capstone domain
differs; the structure stays put.
"""
from __future__ import annotations

import asyncio
import json
import logging

from pydantic import BaseModel, Field

logger = logging.getLogger(__name__)


# ─── Lazy client (W11 hardening) ─────────────────────────────────────────────
def _async_openai(api_key: str | None = None):
    """Create the AsyncOpenAI client on first use, not at import.

    Importing this module must never require an API key — unit tests import it
    with no environment set. The `openai` import is also deferred so a machine
    without the package can still import the rubric constants.
    """
    from openai import AsyncOpenAI

    return AsyncOpenAI(api_key=api_key)


# ─── Output schema ───────────────────────────────────────────────────────────
class JudgeScore(BaseModel):
    """One judge verdict on one candidate answer."""

    accuracy:     int = Field(..., ge=1, le=4)
    groundedness: int = Field(..., ge=1, le=4)
    format:       int = Field(..., ge=1, le=4)
    reasoning:    str = Field(..., min_length=1)

    @property
    def mean(self) -> float:
        """Average across the three dimensions — useful for sorting/convergence."""
        return (self.accuracy + self.groundedness + self.format) / 3.0

    @property
    def task_success(self) -> bool:
        """KPI #1 rule: a 'successful' answer is accuracy>=3 AND groundedness>=3.

        Format alone never makes or breaks task success — a well-formatted but
        wrong answer is still a failure. Tune this bar in your ADR (D5).
        """
        return self.accuracy >= 3 and self.groundedness >= 3


# ─── Rubric prompt — Enterprise Knowledge Assistant capstone ─────────────────
RUBRIC_PROMPT: str = """\
You are a senior evaluator for an Enterprise Knowledge Assistant — an LLM-backed
service that answers employees' questions about company policies, processes,
and documents.

You rate each candidate answer on three dimensions, each on a 1-4 scale.

ACCURACY — Is the answer factually correct given the ideal answer?
  1 — Wrong. States something that contradicts the ideal answer.
  2 — Partially correct. Some facts right, some wrong or missing critical detail.
  3 — Correct. All stated facts align with the ideal; minor omissions acceptable.
  4 — Correct and complete. Captures every fact in the ideal answer.

GROUNDEDNESS — Does every claim trace to the ideal/source, or does the answer
honestly admit uncertainty when relevant?
  1 — Hallucinates. Invents facts not in the source.
  2 — Mostly grounded but adds 1-2 unsupported claims.
  3 — Grounded. All claims trace to the source. May lack citations.
  4 — Grounded and explicit. Claims trace clearly; cites where possible; admits
      uncertainty if the ideal answer would.

FORMAT — Is the structure, length, and presentation right for the question?
  1 — Wrong shape. Too long/short, missing required structure, garbled.
  2 — Functional but rough. Right info, wrong shape.
  3 — Appropriate. Right length, clear structure, reasonable presentation.
  4 — Polished. Length and structure match what a senior writer would produce.

ALWAYS call the rate_answer tool. Never reply in plain text. The reasoning field
should be 2-4 sentences naming what's strong and what's weak.
"""


# ─── Tool schema for structured output ───────────────────────────────────────
RATE_ANSWER_TOOL: dict = {
    "type": "function",
    "function": {
        "name": "rate_answer",
        "description": (
            "Rate a candidate answer on three dimensions, 1-4 each, "
            "and explain your reasoning."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "accuracy":     {"type": "integer", "minimum": 1, "maximum": 4},
                "groundedness": {"type": "integer", "minimum": 1, "maximum": 4},
                "format":       {"type": "integer", "minimum": 1, "maximum": 4},
                "reasoning":    {"type": "string"},
            },
            "required": ["accuracy", "groundedness", "format", "reasoning"],
        },
    },
}


# ─── Judge call ──────────────────────────────────────────────────────────────
async def judge_one(
    question: str,
    candidate_answer: str,
    ideal_answer: str,
    *,
    judge_model: str = "gpt-4o",
    api_key: str | None = None,
) -> JudgeScore:
    """Run the judge on one (question, candidate, ideal) triple."""
    client = _async_openai(api_key)

    user_msg = (
        f"QUESTION:\n{question}\n\n"
        f"IDEAL ANSWER:\n{ideal_answer}\n\n"
        f"CANDIDATE ANSWER:\n{candidate_answer}\n\n"
        "Rate the candidate using the rate_answer tool."
    )

    resp = await client.chat.completions.create(
        model=judge_model,
        messages=[
            {"role": "system", "content": RUBRIC_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        tools=[RATE_ANSWER_TOOL],
        tool_choice={"type": "function", "function": {"name": "rate_answer"}},
    )

    tool_calls = resp.choices[0].message.tool_calls or []
    if not tool_calls:
        raise RuntimeError(
            "Judge did not call the rate_answer tool. Check the rubric prompt."
        )
    args = json.loads(tool_calls[0].function.arguments)
    return JudgeScore.model_validate(args)


def judge_one_sync(
    question: str,
    candidate_answer: str,
    ideal_answer: str,
    *,
    judge_model: str = "gpt-4o",
    api_key: str | None = None,
) -> JudgeScore:
    """Synchronous convenience wrapper around `judge_one`.

    The eval suite (test_kpi.py) is synchronous pytest; this lets it grade
    task success without managing an event loop.
    """
    return asyncio.run(
        judge_one(
            question, candidate_answer, ideal_answer,
            judge_model=judge_model, api_key=api_key,
        )
    )


### Task-Success Judge

The task-success judge evaluates an answer using three dimensions: accuracy, groundedness and format. Each dimension receives a score from 1 to 4.

The mean score is calculated by averaging the three dimensions. However, task success is determined using a separate rule:

- Accuracy must be at least 3.
- Groundedness must be at least 3.

Both conditions must be satisfied. Format does not determine task success.

The demonstration shows that an answer can have a higher average score but still fail if its accuracy is below 3.

**Key takeaway:** A well-presented answer is not necessarily a successful answer. Correctness and source support are essential.

### The task-success rule, exercised locally** (no API call) — KPI #1 counts an answer successful when accuracy ≥ 3 **and** groundedness ≥ 3; format alone never rescues a wrong answer:

In [10]:
good = JudgeScore(accuracy=4, groundedness=3, format=2, reasoning='correct, a bit terse')
bad  = JudgeScore(accuracy=2, groundedness=4, format=4, reasoning='beautifully wrong')
print('good  -> mean', round(good.mean, 2), '| task_success:', good.task_success)
print('bad   -> mean', round(bad.mean, 2),  '| task_success:', bad.task_success)

good  -> mean 3.0 | task_success: True
bad   -> mean 3.33 | task_success: False


**The task-success rule, exercised locally (no API call)**

KPI #1 counts an answer as successful when **accuracy ≥ 3 AND groundedness ≥ 3**. Format alone never rescues a wrong answer.

The `JudgeScore` model calculates two separate outputs:

1. **Mean score:** The average of accuracy, groundedness and format.
2. **Task success:** A Boolean result (`True` or `False`) based only on the accuracy and groundedness thresholds.

The two sample cases demonstrate why these outputs must be interpreted separately:

- **Good answer:** Accuracy = 4, groundedness = 3, format = 2. Mean = 3.0. Task success = True.
- **Bad answer:** Accuracy = 2, groundedness = 4, format = 4. Mean = 3.33. Task success = False.

Although the second answer has a higher mean, it fails the task-success rule because its accuracy is below 3.

**Key takeaway:** A well-formatted answer with incorrect information must not be counted as a successful answer.

## Recap

Four numbers, one pass: **faithfulness** and **answer relevance** read on the *answer*; **context precision** and **context recall** read on *retrieval*. A low score is a **first hypothesis** about which layer to inspect — confirm before you fix (a low answer metric often has a retrieval root cause). Next, **Notebook 2** turns these into the 8-KPI dashboard, persists per-query cost/latency to `results.db`, and sets thresholds you can defend.